In [ ]:
!pip install -q datasets
!pip install -q peft
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q evaluate
!pip install -q rouge_score

In [ ]:

import torch
import json
import os

from datasets import Dataset

from transformers import (
    MBartForConditionalGeneration,
    MBart50TokenizerFast,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType
)

print(torch.cuda.is_available())
!nvidia-smi
with open(
    "/content/training_dataset.json",
    "r",
    encoding="utf-8"
) as f:
    train_data = json.load(f)


len(train_data)

True
Fri Jul 31 10:25:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   77C    P0             34W /   70W |    5175MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+

2292

In [ ]:


def create_phase1_prompt(example):

    text = f"""
Context:

<Article>
Article Number: {example['article_number']}
Title: {example['article_title']}
Text:
{example['article_text']}
</ARTICLE>


Question:
{example['question']}
"""

    return {
        "input_text": text.strip(),
        "target_text": example["answer"]
    }


phase1_dataset = [
    create_phase1_prompt(x)
    for x in train_data
]



print(phase1_dataset[5])
dataset = Dataset.from_list(
    phase1_dataset
)


dataset


{'input_text': "Context:\n\n<Article>\nArticle Number: 1\nTitle: The Republic\nText:\nBangladesh is a unitary, independent, sovereign Republic to be known as the People's Republic of Bangladesh.\n</ARTICLE>\n\n\nQuestion:\nBy what name is Bangladesh to be known?", 'target_text': "It is to be known as the People's Republic of Bangladesh."}


Dataset({
    features: ['input_text', 'target_text'],
    num_rows: 2292
})

In [ ]:
model_name = "facebook/mbart-large-50-many-to-many-mmt"


tokenizer = MBart50TokenizerFast.from_pretrained(
    model_name,
    src_lang="en_XX",
    tgt_lang="en_XX"
)
max_input_length = 512
max_target_length = 128


def tokenize(batch):

    model_inputs = tokenizer(
        batch["input_text"],
        truncation=True,
        max_length=max_input_length
    )

    labels = tokenizer(
        text_target=batch["target_text"],
        truncation=True,
        max_length=max_target_length
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs


tokenized_dataset = dataset.map(
    tokenize,
    batched=True,
    remove_columns=dataset.column_names
)



model = MBartForConditionalGeneration.from_pretrained(model_name)

from peft import LoraConfig, TaskType


lora_config = LoraConfig(

    r=16,

    lora_alpha=32,

    target_modules=[
        "q_proj",
        "v_proj"
    ],

    lora_dropout=0.05,

    bias="none",

    task_type=TaskType.SEQ_2_SEQ_LM
)
!pip uninstall -y torchao
import torch
import peft
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
model = get_peft_model(
    model,
    lora_config
)


model.print_trainable_parameters()
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8
)
training_args = Seq2SeqTrainingArguments(

    output_dir="./mbart_phase1",

    num_train_epochs=8,

    per_device_train_batch_size=2,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    warmup_ratio=0.05,

    weight_decay=0.01,

    fp16=True,

    logging_steps=25,

    save_strategy="epoch",

    save_total_limit=3,

    predict_with_generate=True,

    generation_max_length=128,

    report_to="none"
)
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    data_collator=data_collator
)

Map:   0%|          | 0/2292 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/516 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Torch: 2.11.0+cu128
Transformers: 5.13.1
PEFT: 0.19.1
trainable params: 2,359,296 || all params: 613,238,784 || trainable%: 0.3847


In [ ]:
trainer.train()

Step,Training Loss
25,24.839617
50,16.116409
75,10.299385
100,8.821680
125,8.050911
150,7.139455
175,7.331923
200,7.068298
225,6.813262
250,6.802972


TrainOutput(global_step=1152, training_loss=6.344199671513504, metrics={'train_runtime': 1253.2116, 'train_samples_per_second': 14.631, 'train_steps_per_second': 0.919, 'total_flos': 9406588724183040.0, 'train_loss': 6.344199671513504, 'epoch': 8.0})

In [ ]:
save_path = "/content/mbart_phase1_lora"

model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)

print("Saved:", save_path)

Saved: /content/mbart_phase1_lora


In [ ]:
def generate_answer(context, question):

    prompt = f"""
Context:

{context}

Question:
{question}
"""

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)


    with torch.no_grad():

        output_ids = model.generate(
            **inputs,
            max_length=128,
            num_beams=5,
            no_repeat_ngram_size=3,
            length_penalty=2.0,
            early_stopping=True
        )


    return tokenizer.decode(
        output_ids[0],
        skip_special_tokens=True
    )

In [ ]:
context = """
<Article>
Article Number: 3
Title: The state language
Text:
   The capital of the Republic is Dhaka.

</ARTICLE>
"""


question = "Capital city?"


answer = generate_answer(
    context,
    question
)


print(answer)

Dhaka is the capital.


In [ ]:
import os

os.listdir("./mbart_phase1/checkpoint-1152")

['adapter_config.json',
 'optimizer.pt',
 'rng_state.pth',
 'trainer_state.json',
 'adapter_model.safetensors',
 'scaler.pt',
 'tokenizer_config.json',
 'scheduler.pt',
 'training_args.bin',
 'README.md',
 'tokenizer.json']

In [ ]:
import shutil
from google.colab import files

shutil.make_archive(
    "mbart_phase1_checkpoint_1152",
    "zip",
    "./mbart_phase1/checkpoint-1152"
)

files.download("mbart_phase1_checkpoint_1152.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>